In [25]:
import pandas as pd
import numpy as np
train=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\House_Price_Prediction\Kaggles_data\train.csv")
test=pd.read_csv(r"C:\Users\Acer\Desktop\Data_Science_Projects\House_Price_Prediction\Kaggles_data\test.csv")
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [26]:
print(train.shape)
print(test.shape)

(1460, 81)
(1459, 80)


In [27]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [28]:
#Remove outliers
train=train[train['GrLivArea']<4000]

In [29]:
#identify target and features
y=np.log1p(train['SalePrice'])
X=train.drop('SalePrice',axis=1,inplace=True)

In [30]:
#combine train+test
full=pd.concat([train,test],axis=0)

# Feature Engineering

In [31]:
#total Square feet
full['TotalSf']=(
    full['TotalBsmtSF']+full['1stFlrSF']+full['2ndFlrSF']
)

In [32]:
#Total Bathrooms
full['TotalBath']=(
    full['FullBath']+0.5*full['HalfBath']+full['BsmtFullBath']+0.5*full['BsmtHalfBath']
)

In [33]:
#Age features
full['HouseAge']=full['YrSold']-full['YearBuilt']
full['RemodAge']=full['YrSold']-full['YearRemodAdd']

In [34]:
#Binary Features
full['HasGarage']=(full['GarageArea']>0).astype(int)
full['HasBsmt']=(full['TotalBsmtSF']>0).astype(int)
full['HasPool']=(full['PoolArea']>0).astype(int)

In [35]:
from sklearn.model_selection import KFold,cross_val_score
from scipy.stats import skew

In [36]:
#Fix Skewed Numerical Features
numeric_f=full.select_dtypes(include=[np.number]).columns
skewed=full[numeric_f].apply(lambda x: skew(x.dropna()))
skewed=skewed[skewed>0.75].index
full[skewed]=np.log1p(full[skewed])

In [37]:
#Handle missing values
for col in full.select_dtypes(include=[np.number]).columns:
    full[col]=full[col].fillna(full[col].median())

In [38]:
for col in full.select_dtypes(include=['object']).columns:
    full[col]=full[col].fillna("None")

In [39]:
#One Hot Encode
full=pd.get_dummies(full)

In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error

In [41]:
#Split Back
X=full.iloc[:len(train)]
X_test=full.iloc[len(train):]

In [42]:
from xgboost import XGBRegressor
xgb_model=XGBRegressor(
    n_estimators=3000,
    random_state=42,
    learning_rate=0.025,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.75,
    reg_alpha=0.3,
    reg_lambda=1.5,
    n_jobs=-1
)


In [43]:
kf=KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
scores=cross_val_score(
    xgb_model,
    X,
    y,
    scoring='neg_root_mean_squared_error',
    cv=kf
)
print("Fold RMSE scores : ")
print(-scores)
print("Mean CV RMSE: ")
print(-scores.mean())

Fold RMSE scores : 
[0.12821319 0.10985589 0.13108722 0.1273433  0.10487754]
Mean CV RMSE: 
0.12027542763030878


In [44]:
xgb_model.fit(X,y)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.75
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [45]:
pred=xgb_model.predict(X_test)
pred=np.expm1(pred)

In [46]:
#Create submission file
submission = pd.DataFrame({
    'ID':test['Id'],
    'SalePrice':pred
})
submission.to_csv("submission3.csv",index=False)